In [7]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 40.5 MB/s eta 0:00:00


In [2]:
import json
from collections import Counter, defaultdict
import os
import subprocess
from google.colab import drive
from gensim.models import KeyedVectors

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
CORPUS_PATH = "/content/drive/MyDrive/InformationRetrieval/Data/clean_corpus.jsonl"
QUERIES_PATH = "/content/drive/MyDrive/InformationRetrieval/Data/clean_queries.jsonl"

# Link tải BioWordVec (13GB)
BIOWORDVEC_URL = "https://ftp.ncbi.nlm.nih.gov/pub/lu/Suppl/BioSentVec/BioWordVec_PubMed_MIMICIII_d200.vec.bin"
MODEL_PATH = "/content/BioWordVec_PubMed_MIMICIII_d200.vec.bin"

TOP_ENTITIES = 30

In [10]:
def download_biowordvec():
    if os.path.exists(MODEL_PATH):
        print(f"File đã tồn tại: {MODEL_PATH}")
        return

    print("Đang tải BioWordVec (13GB)...")

    # Tải bằng wget với progress bar
    subprocess.run([
        "wget", "-O", MODEL_PATH, BIOWORDVEC_URL
    ], check=True)

    print("Tải xong!")

In [11]:
download_biowordvec()

Đang tải BioWordVec (13GB)...
Tải xong!


In [4]:
def load_jsonl(path: str) -> dict:
    """
    Đọc file .jsonl, trả về dict {id: [token, ...]}
    Mỗi dòng có dạng: {"_id": "...", "text": "word1 word2 ..."}
    """
    data = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            tokens = obj["text"].split()          # text đã tokenize sẵn
            data[obj["_id"]] = tokens
    return data

In [5]:
def classify_token(token: str, model_vocab: set) -> str:
    """
    Trả về 1 trong 4 nhãn:
      in_vocab        – token đơn có trong BioWordVec
      merged_full     – token ghép (breast_cancer), TẤT CẢ parts có trong vocab
      merged_partial  – token ghép, CHỈ MỘT SỐ parts có trong vocab
      true_oov        – không tìm được vector theo bất kỳ cách nào
    """
    t = token.lower()

    if t in model_vocab:
        return "in_vocab"

    if "_" in t:
        parts = [p for p in t.split("_") if p]  # tránh phần rỗng
        if not parts:
            return "true_oov"
        found = sum(1 for p in parts if p in model_vocab)
        if found == len(parts):
            return "merged_full"
        elif found > 0:
            return "merged_partial"
        else:
            return "true_oov"
    return "true_oov"

In [6]:
def analyze_coverage(token_lists: dict, model_vocab: set, label: str):
    type_counter = Counter()      # đếm unique token theo loại
    token_counter = Counter()     # đếm theo occurrence
    examples = defaultdict(set)

    for tokens in token_lists.values():
        for tok in tokens:
            if not tok:
                continue
            label_tok = classify_token(tok, model_vocab)
            token_counter[label_tok] += 1

            lower_tok = tok.lower()
            if lower_tok not in [item for sublist in examples.values() for item in sublist]:  # heuristic đơn giản
                if len(examples[label_tok]) < 10:
                    examples[label_tok].add(lower_tok)
                type_counter[label_tok] += 1   # chỉ đếm 1 lần cho type

    total_types = sum(type_counter.values())
    total_tokens = sum(token_counter.values())

    print(f"\n{'═'*70}")
    print(f" Coverage Report — {label.upper()}")
    print(f"{'═'*70}")
    print(f" {'Nhóm':<20} {'Types':>8} {'%Types':>8} {'Tokens':>10} {'%Tokens':>8}")
    print(f" {'─'*20} {'─'*8} {'─'*8} {'─'*10} {'─'*8}")

    order = ["in_vocab", "merged_full", "merged_partial", "true_oov"]
    for key in order:
        tc = type_counter.get(key, 0)
        oc = token_counter.get(key, 0)
        tp = tc / total_types * 100 if total_types else 0
        op = oc / total_tokens * 100 if total_tokens else 0
        print(f" {key:<20} {tc:>8,} {tp:>7.1f}% {oc:>10,} {op:>7.1f}%")

    print(f" {'─'*20} {'─'*8} {'─'*8} {'─'*10} {'─'*8}")
    print(f" {'TOTAL':<20} {total_types:>8,} {'100.0%':>8} {total_tokens:>10,} {'100.0%':>8}")

    recoverable_types = sum(type_counter.get(k, 0) for k in ["in_vocab", "merged_full", "merged_partial"])
    recoverable_tokens = sum(token_counter.get(k, 0) for k in ["in_vocab", "merged_full", "merged_partial"])

    print(f"\n Recoverable (có thể lấy vector):")
    print(f"    Types : {recoverable_types:,} / {total_types:,} ({recoverable_types/total_types*100:.1f}%)")
    print(f"    Tokens: {recoverable_tokens:,} / {total_tokens:,} ({recoverable_tokens/total_tokens*100:.1f}%)")

    print(f"\n Ví dụ (tối đa 10 mỗi nhóm):")
    for key in order:
        ex = sorted(examples[key])[:10]
        print(f"  [{key}]: {', '.join(ex) if ex else '—'}")

    return {"types": type_counter, "tokens": token_counter}

In [15]:
model = KeyedVectors.load_word2vec_format(MODEL_PATH, binary=True, unicode_errors='ignore', limit = 2000000)
vocab = set(model.key_to_index.keys())
print(f"→ Vocab size: {len(vocab):,} terms | Vector dim: {model.vector_size}d")

→ Vocab size: 2,000,000 terms | Vector dim: 200d


In [16]:
print("\nĐang load corpus và queries...")
corpus_data = load_jsonl(CORPUS_PATH)
queries_data = load_jsonl(QUERIES_PATH)

print(f"→ Corpus : {len(corpus_data):,} documents")
print(f"→ Queries: {len(queries_data):,} queries")


Đang load corpus và queries...
→ Corpus : 3,633 documents
→ Queries: 3,237 queries


In [17]:
analyze_coverage(corpus_data, vocab, "CORPUS")
analyze_coverage(queries_data, vocab, "QUERIES")


══════════════════════════════════════════════════════════════════════
 Coverage Report — CORPUS
══════════════════════════════════════════════════════════════════════
 Nhóm                    Types   %Types     Tokens  %Tokens
 ──────────────────── ──────── ──────── ────────── ────────
 in_vocab              299,147    79.6%    304,530    79.7%
 merged_full            68,935    18.4%     69,596    18.2%
 merged_partial          1,358     0.4%      1,360     0.4%
 true_oov                6,194     1.6%      6,690     1.8%
 ──────────────────── ──────── ──────── ────────── ────────
 TOTAL                 375,634   100.0%    382,176   100.0%

 Recoverable (có thể lấy vector):
    Types : 369,440 / 375,634 (98.4%)
    Tokens: 375,486 / 382,176 (98.2%)

 Ví dụ (tối đa 10 mỗi nhóm):
  [in_vocab]: cardiovascular, delay, effect, mortality, prevent, prevention, recurrence, risk, statin, unclear
  [merged_full]: breast_cancer, cox_proportional_hazard_regression_method, disease-specific_mortali

{'types': Counter({'merged_full': 1443,
          'in_vocab': 4614,
          'true_oov': 71,
          'merged_partial': 40}),
 'tokens': Counter({'merged_full': 1504,
          'in_vocab': 4813,
          'true_oov': 87,
          'merged_partial': 46})}